Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Analytical charts: trends, smoothing, fits and distributions

A chart can do more than show the data: it can **analyse** it. This notebook covers the techniques from the lecture's section 9.2, one at a time. For each, we do the analysis in **pandas** (so you can see the numbers), then make the chart in **Altair**, and show the Vega-Lite transform that does the same job inside the chart.

In this notebook:
1. **Rebasing:** making series with different levels comparable
2. **Small multiples** with independent y-axes
3. **Moving averages:** smoothing a noisy series
4. **Trends:** fitting a trend, and the gaps from it
5. **Fitted lines on a scatter:** regression and loess
6. **Distributions:** histogram, density and box plot
7. **Event studies:** what happened after a shock, compared with the trend before it
8. **Uncertainty:** estimates with error bars

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd
import altair as alt

DATA = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week09/data/"
MENU = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week08/data/"     # Week 8's menu prices
MAPS = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/"

SOURCE = "Source: LSE Growth Lab, AutoCPI (menu photos)"     # for chart subtitles
BROWN, BAND = "#8c5a2b", "#ecdfd0"     # colours for the menu charts

import numpy as np

<br><br>

---

<br>

## 1. Rebasing

Prices of different dishes are hard to compare: a latte costs about £4, fish & chips about £14. **Rebasing** divides each series by its value in a **base year**, times 100. Every series is then 100 in the base year, and the lines show *percentage* changes.

The Week 8 menu prices, summarised by year:

In [ ]:
prices = pd.read_csv(MENU + "menu_percentiles_by_year.csv")
prices["date"] = pd.to_datetime(prices["year"], format="%Y")
prices.head()

<br>

Each dish's 2019 price, then divide. `.map(base)` looks up each row's dish in `base`:

In [ ]:
base = prices[prices["year"] == 2019].set_index("item")["median"]
prices["index"] = prices["median"] / prices["item"].map(base) * 100

four = ["latte", "fish & chips", "margherita", "guinness"]
some = prices[prices["item"].isin(four)]
some.pivot(index="year", columns="item", values="index").round(0)

<br>

A dashed line at 100 marks the base year. `alt.datum(100)` draws a line at a fixed value rather than a column:

In [ ]:
lines = alt.Chart(some).mark_line(point=True).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("index:Q").title("Index, 2019 = 100").scale(zero=False),
    color=alt.Color("item:N").title(None),
    tooltip=[
        alt.Tooltip("item:N").title("Dish"),
        alt.Tooltip("year:O").title("Year"),
        alt.Tooltip("median:Q").format(".2f").title("Median (£)"),
        alt.Tooltip("index:Q").format(".0f").title("Index (2019 = 100)"),
    ],
)
hundred = alt.Chart().mark_rule(strokeDash=[4, 3], color="#999").encode(y=alt.datum(100))

(hundred + lines).properties(width=450, height=280,
                             title=alt.Title("Which dishes rose most?", subtitle="Median menu price, 2019 = 100. " + SOURCE))

<br>

> **In Vega-Lite:** the same steps as transforms: a `calculate` keeps the 2019 price, a `joinaggregate` copies it to every row of that dish, a second `calculate` divides. See `examples/analysis/A1_rebasing.json`. Doing it in the chart means a slider can change the base year (`examples/lecture/L4_rebase_slider.json`).

<br><br>

---

<br>

## 2. Small multiples with independent y-axes

A **trellis** (small multiples) draws one panel per group. By default the panels share a y-axis, which is right when the levels are similar. When they're not, `resolve_scale(y="independent")` gives each panel its own:

In [ ]:
band = alt.Chart().mark_area(color="#f2c4c4").encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("p10:Q").title("£"),
    y2="p90:Q",
)
line = alt.Chart().mark_line(color="#c8102e").encode(
    x="date:T",
    y="median:Q",
)

alt.layer(band, line, data=some).properties(width=200, height=130).facet(
    alt.Facet("item:N").title(None).sort(four), columns=2,
).resolve_scale(y="independent").properties(
    title=alt.Title("Menu prices: median and the middle 80%", subtitle="Each panel has its own scale. " + SOURCE))

<br>

Try removing `.resolve_scale(...)`: the latte and the pint become flat lines at the bottom of their panels. Independent axes show each dish's *shape*; a shared axis shows their *levels*. Say which in the subtitle.

<br><br>

---

<br>

## 3. Moving averages

Daily data is noisy. A **moving average** replaces each day with the mean of the days around it. In pandas, `rolling(31, center=True)` takes 31 days centred on each one (15 before, 15 after):

In [ ]:
bitcoin = pd.read_csv(DATA + "bitcoin_price.csv", encoding="utf-8-sig", usecols=["Date", "Price"], parse_dates=["Date"])
bitcoin = bitcoin.sort_values("Date")                # rolling works down the rows, so sort by date first
bitcoin["ma31"] = bitcoin["Price"].rolling(31, center=True).mean()

recent = bitcoin[bitcoin["Date"] >= "2020-01-01"]
print(f"{len(recent):,} days")
recent.tail()

<br>

`encoding="utf-8-sig"` drops an invisible character at the start of this file (a *byte order mark*). Without it, the first column is called `\ufeffDate`.

The last 15 days have no average: there aren't 15 days after them. Grey for the daily price, a bold line for the average:

In [ ]:
daily = alt.Chart(recent).mark_line(color="#bbbbbb", strokeWidth=1).encode(
    x=alt.X("Date:T").title(None),
    y=alt.Y("Price:Q").title("US$").axis(format="$,.0f"),
)
smooth = alt.Chart(recent).mark_line(color="#f2a900", strokeWidth=2.5).encode(
    x="Date:T",
    y="ma31:Q",
    tooltip=[
        alt.Tooltip("Date:T").format("%d %b %Y").title("Date"),
        alt.Tooltip("Price:Q").format("$,.0f").title("Price"),
        alt.Tooltip("ma31:Q").format("$,.0f").title("31-day average"),
    ],
)
(daily + smooth).properties(width=500, height=280,
                            title=alt.Title("Bitcoin: smoothing a noisy series",
                                            subtitle="Daily price (grey) and a centred 31-day moving average. Source: Investing.com"))

<br>

> **In Vega-Lite:** a `window` transform with `frame`: `transform_window(ma31="mean(Price)", sort=[alt.SortField("Date")], frame=[-15, 15])`. `frame=[-30, 0]` is a **trailing** average (only past days), which is what you'd use for a chart updated every day. See `examples/analysis/A3_moving_average.json`.

<br><br>

---

<br>

## 4. Trends, and the gaps from them

Over decades, GDP per person grows at a roughly constant **rate**. A constant growth rate is a straight line in **logs**. So fit a straight line to log GDP per person: the fitted line is the **trend**, and the gaps from it are the **business cycle**.

US real GDP per person, quarterly:

In [ ]:
us = pd.read_csv(DATA + "us_real_gdp_per_capita.csv", encoding="utf-8-sig", parse_dates=["Date"])
us["t"] = us["Date"].dt.year + (us["Date"].dt.month - 1) / 12       # time in years: 1947.0, 1947.25, ...

slope, intercept = np.polyfit(us["t"], np.log(us["Value"]), 1)      # a straight line through log GDP
us["trend"] = np.exp(intercept + slope * us["t"])
us["gap"] = us["Value"] / us["trend"] - 1                          # % above (+) or below (-) trend

print(f"Trend growth: {np.exp(slope) - 1:.1%} a year")
us.tail()

<br>

`np.polyfit(x, y, 1)` fits a straight line (a polynomial of degree 1) by least squares and returns the slope and intercept. The slope of a line in logs is a growth rate.

Two charts, stacked with `&`: the level with its trend (log scale), and the gap:

In [ ]:
level = alt.Chart(us).mark_line(color="#1f4e79").encode(
    x=alt.X("Date:T").title(None),
    y=alt.Y("Value:Q").title("US$ (2017 prices)").scale(type="log", domain=[12000, 70000], nice=False).axis(format="$,.0f"),
)
trend = alt.Chart(us).mark_line(color="#999999", strokeDash=[5, 4]).encode(
    x="Date:T",
    y="trend:Q",
)
gap = alt.Chart(us).mark_area(color="#1f4e79", opacity=0.8).encode(
    x=alt.X("Date:T").title(None),
    y=alt.Y("gap:Q").title("Gap from trend").axis(format="+.0%"),
    tooltip=[
        alt.Tooltip("Date:T").format("%b %Y").title("Quarter"),
        alt.Tooltip("gap:Q").format("+.1%").title("Gap from trend"),
    ],
)

((level + trend).properties(width=500, height=200) & gap.properties(width=500, height=130)).properties(
    title=alt.Title("US GDP per person: trend and cycle",
                    subtitle="Top: GDP per person and its log-linear trend. Bottom: % above or below trend. Source: FRED"))

<br>

The result depends on the trend you choose. Fitted over 1947 to 2023, the US has been below trend ever since 2008: is that a long slump, or a slower trend? Try fitting only on 1947 to 2007 (`us[us["t"] < 2008]`) and extending it.

> **In Vega-Lite:** `transform_regression("Value", "Date", method="exp")` draws the trend line. For the gaps you need the fitted values on every row: `examples/analysis/A4_trend_deviations.json` does the least squares with two `joinaggregate` transforms.

<br><br>

---

<br>

## 5. Fitted lines on a scatter

Do richer US states have fewer gun deaths? First, the numbers: the slope of a straight-line fit, and the correlation:

In [ ]:
states = pd.read_csv(DATA + "us_states_socioeconomic.csv")

slope, intercept = np.polyfit(states["medIncome"], states["DeathRate"], 1)
r = states["medIncome"].corr(states["DeathRate"])
print(f"Each extra $10,000 of median income: {slope * 10000:+.1f} firearm deaths per 100,000")
print(f"Correlation: {r:.2f}")

<br>

Altair fits lines itself. `transform_regression` fits a straight line (or `method="log"`, `"exp"`, `"poly"`...); `transform_loess` fits a smooth curve that follows the data locally. `bandwidth` sets how local: smaller is wigglier. Each fit is a layer over the points:

In [ ]:
points = alt.Chart(states).mark_circle(size=70, color="#555555", opacity=0.7).encode(
    x=alt.X("medIncome:Q").title("Median household income (US$)").scale(zero=False).axis(format="$,.0f"),
    y=alt.Y("DeathRate:Q").title("Firearm deaths per 100,000"),
    tooltip=[
        alt.Tooltip("State:N").title("State"),
        alt.Tooltip("medIncome:Q").format("$,.0f").title("Median income"),
        alt.Tooltip("DeathRate:Q").format(".1f").title("Firearm deaths per 100k"),
    ],
)
linear = points.transform_regression("medIncome", "DeathRate").mark_line(color="#c8102e", strokeDash=[6, 3])
loess = points.transform_loess("medIncome", "DeathRate", bandwidth=0.75).mark_line(color="#1f77b4")

(points + linear + loess).properties(width=450, height=320,
                                     title=alt.Title("Richer states have fewer gun deaths",
                                                     subtitle="Red dashed: straight-line fit. Blue: loess. Source: CDC, US Census Bureau"))

<br>

The loess shows what the straight line hides: the relationship is steep up to about $60,000, then flattens. And a fitted line shows a **correlation**, not a cause: richer states differ in many other ways too (urban, gun laws, ...). That's what the lecture's causal methods are for.

> Note the order in the transforms: `transform_regression("medIncome", "DeathRate")` is `(on, regress)`, x first. In Vega-Lite JSON it's `{"regression": "DeathRate", "on": "medIncome"}`.

<br><br>

---

<br>

## 6. Distributions

Life expectancy in 145 countries. Three ways to show how the values are spread out, from detailed to compact:

In [ ]:
countries = pd.read_csv(DATA + "gdp_life_expectancy.csv")
countries = countries[countries["Continent"] != "Oceania"]        # only two countries
countries.groupby("Continent")["Life Expectancy"].describe().round(1)

<br>

**A histogram:** `bin` groups the values into bins, and `count()` counts each one:

In [ ]:
x = alt.X("Life Expectancy:Q").title("Life expectancy (years)").scale(domain=[50, 90])

histogram = alt.Chart(countries).mark_bar(color="#888888").encode(
    x=x.bin(step=2, extent=[50, 90]),
    y=alt.Y("count():Q").title("Countries"),
).properties(width=450, height=130, title="Histogram")
histogram

<br>

**A density curve** smooths the histogram. `groupby` draws one curve per continent. `stack=None` lets the curves overlap; by default, areas stack on top of each other:

In [ ]:
colours = alt.Scale(domain=["Africa", "Asia", "America", "Europe"], range=["#e45756", "#f58518", "#54a24b", "#4c78a8"])

density = alt.Chart(countries).transform_density(
    "Life Expectancy", groupby=["Continent"], extent=[50, 90], as_=["years", "density"],
).mark_area(opacity=0.45, line=True).encode(
    x=alt.X("years:Q").title("Life expectancy (years)").scale(domain=[50, 90]),
    y=alt.Y("density:Q").title("Density").stack(None).axis(labels=False, ticks=False),
    color=alt.Color("Continent:N").title(None).scale(colours),
).properties(width=450, height=130, title="Density, by continent")
density

<br>

**A box plot** puts each group in one line: the box is the middle half (25th to 75th percentile), the line in it the median, and the whiskers the range (`extent="min-max"`):

In [ ]:
boxplot = alt.Chart(countries).mark_boxplot(extent="min-max", size=18).encode(
    x=x,
    y=alt.Y("Continent:N").title(None).sort(["Africa", "Asia", "America", "Europe"]),
    color=alt.Color("Continent:N").scale(colours).legend(None),
).properties(width=450, height=130, title="Box plot, by continent")

(histogram & density & boxplot).properties(
    title=alt.Title("Life expectancy around the world", subtitle="Source: World Bank, World Happiness Report"))

<br>

The histogram shows the detail (two peaks); the density compares shapes; the box plot compares centres and spreads at a glance, but hides the shape: Africa's long tail of countries in the 50s is just a whisker.

<br><br>

---

<br>

## 7. Event studies

An event study asks: what happened after an event, compared with what we'd have expected? The **counterfactual** is often the trend before the event, extended.

UK productivity (output per hour worked) since 1997, from the ONS:

In [ ]:
productivity = pd.read_csv(DATA + "uk_productivity.csv")
productivity["date"] = pd.to_datetime(productivity["year"], format="%Y")

pre = productivity[productivity["year"] <= 2007]                          # before the financial crisis
slope, intercept = np.polyfit(pre["year"], np.log(pre["output_per_hour"]), 1)
productivity["trend"] = np.exp(intercept + slope * productivity["year"])    # the pre-crisis trend, for every year
productivity["gap"] = 1 - productivity["output_per_hour"] / productivity["trend"]

print(f"Pre-crisis trend growth: {np.exp(slope) - 1:.1%} a year")
print(f"2025: {productivity['gap'].iloc[-1]:.0%} below the pre-crisis trend")

<br>

The fit uses only 1997 to 2007; the trend is then calculated for every year. Shade the gap after 2007 with an area between two columns (`y` and `y2`):

In [ ]:
after = productivity[productivity["year"] >= 2007]

gap_area = alt.Chart(after).mark_area(color="#f4c2c2", opacity=0.7).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("output_per_hour:Q"),
    y2="trend:Q",
)
trend_line = alt.Chart(productivity).mark_line(color="#999999", strokeDash=[6, 4]).encode(
    x="date:T",
    y="trend:Q",
)
actual = alt.Chart(productivity).mark_line(color="#c8102e", point=alt.OverlayMarkDef(color="#c8102e")).encode(
    x="date:T",
    y=alt.Y("output_per_hour:Q").title("Output per hour (2024 = 100)").scale(zero=False),
    tooltip=[
        alt.Tooltip("year:O").title("Year"),
        alt.Tooltip("output_per_hour:Q").format(".1f").title("Output per hour"),
        alt.Tooltip("trend:Q").format(".1f").title("Pre-crisis trend"),
        alt.Tooltip("gap:Q").format(".0%").title("Below trend"),
    ],
)

(gap_area + trend_line + actual).properties(width=480, height=300,
    title=alt.Title("The UK's productivity puzzle",
                    subtitle="Output per hour worked. Dashed: the 1997-2007 trend, extended. Source: ONS (LZVB)"))

<br>

The gap depends on assumptions: would the old trend really have continued? The pre-crisis years included a boom in finance, which may not have lasted anyway. (And the jump in 2020 isn't a productivity miracle: in lockdown, lower-paid hours fell most, which raised the average.) An event study is only as good as its counterfactual: the lecture's **difference-in-differences** uses a comparison group instead.

<br><br>

---

<br>

## 8. Uncertainty: estimates with error bars

A difference-in-differences study from the Economics Observatory: what happened to youth offending after youth clubs closed? Each point is an estimate for one year before or after closure, compared with the year before (year -1, set to zero). The 95% **confidence interval** shows the uncertainty:

In [ ]:
effects = pd.read_csv(DATA + "youth_clubs_crime_effects.csv")
stacked = effects[effects["Estimates"] == "Stacked"]
stacked

<br>

An error bar is a `rule` from `y` (lower) to `y2` (upper), with the estimate as a point on top. A rule at zero is the "no effect" line:

In [ ]:
bars = alt.Chart(stacked).mark_rule(strokeWidth=2, color="#676a86").encode(
    x=alt.X("Year:Q").title("Years since the youth club closed").axis(tickMinStep=1).scale(domain=[-4.5, 5.5], nice=False),
    y=alt.Y("CI95_lower:Q").title("Effect on youth offending"),
    y2="CI95_upper:Q",
)
estimates = alt.Chart(stacked).mark_circle(size=80, color="#676a86", opacity=1).encode(
    x="Year:Q",
    y="Beta:Q",
    tooltip=[
        alt.Tooltip("Year:Q").title("Years since closure"),
        alt.Tooltip("Beta:Q").format(".2f").title("Estimate"),
        alt.Tooltip("CI95_lower:Q").format(".2f").title("95% CI, lower"),
        alt.Tooltip("CI95_upper:Q").format(".2f").title("95% CI, upper"),
    ],
)
zero = alt.Chart().mark_rule(color="#999999", strokeDash=[4, 3]).encode(y=alt.datum(0))

(zero + bars + estimates).properties(width=450, height=280,
    title=alt.Title("Youth offending rose after youth clubs closed",
                    subtitle="Difference-in-differences estimates with 95% confidence intervals. Source: Economics Observatory"))

<br>

Before closure (years -4 to -2), the estimates are around zero, though year -3 is just below it: little difference between the areas that lost a club and those that didn't. That's the **parallel trends** check from the lecture. From a year after closure (year 0 is the year it closed), the estimates rise and their intervals stay above zero.

> More uncertainty charts, including `errorband` and the OBR's fan chart, are in `examples/uncertainty/`.

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1. A different base year.** Rebase the menu prices to 2017 instead of 2019. Does the ranking of dishes change?

**2. A trailing average.** Remake the Bitcoin chart with a 7-day and a 90-day trailing average (`rolling(7)`, `rolling(90)`, without `center`). Which would you use on a dashboard, and why?

**3. A different trend.** In section 4, fit the US trend on 1947 to 2007 only, and extend it. How far below the old trend is the US now?

**4. A fitted line, your choice.** In `gdp_life_expectancy.csv`, plot life expectancy against GDP per person (`GDP per capita (PPP)`) on a log x-axis, with a straight-line fit and a loess. Then try `transform_regression(..., method="log")` on a linear axis.

**5. Two methods.** The youth clubs file has a second set of estimates (`TWFE`). Show both on one chart, side by side: `examples/uncertainty/DS9_21_error_bars_two_methods.json` shows how, using `xOffset`.

In [ ]:
# Your code here
